# Workload-Based Cost Analysis

This notebook analyzes cost-effectiveness from a **realistic workload perspective** showing scenarios where CPU configurations make economic sense.

**Key Metrics:**
1. Cost per Request (10 requests/hour scenario)
2. Break-even Analysis across request volumes
3. Idle time costs and utilization efficiency
4. Total Cost of Ownership analysis

**Focus Model**: `01-DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M.gguf`

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")

In [ ]:
# Enhanced cost assumptions with idle costs
cost_assumptions = {
    ('cpu', 1): {'hourly_cost': 0.05, 'idle_multiplier': 1.0},
    ('cpu', 2): {'hourly_cost': 0.10, 'idle_multiplier': 1.0},
    ('cpu', 4): {'hourly_cost': 0.20, 'idle_multiplier': 1.0},
    ('cpu', 8): {'hourly_cost': 0.40, 'idle_multiplier': 1.0},
    ('cuda', 25): {'hourly_cost': 0.50, 'idle_multiplier': 0.8},
    ('cuda', 50): {'hourly_cost': 1.00, 'idle_multiplier': 0.8},
    ('cuda', 75): {'hourly_cost': 1.50, 'idle_multiplier': 0.8},
    ('cuda', 100): {'hourly_cost': 2.00, 'idle_multiplier': 0.8},
}

def get_device_key(record):
    variant = record['variant']
    if variant == 'cpu':
        return ('cpu', record['cpu_cores'])
    elif variant == 'cuda':
        return ('cuda', record['gpu_percentage'])

print("Cost assumptions defined with idle cost multipliers")
print("CPU idle multiplier: 1.0 (full cost when idle)")
print("GPU idle multiplier: 0.8 (slightly reduced cost when idle)")

In [ ]:
# Load and filter data for Q4_K_M models
with open('parsed_logs/night_logs_6_with_model_info.json', 'r') as f:
    data = json.load(f)

# Filter for Q4_K_M quantized models
q4_models = [record for record in data if record.get('model_quant', '') == 'Q4_K_M']
print(f"Found {len(q4_models)} Q4_K_M quantized model records")

# Process the data
processed_data = []
for record in q4_models:
    try:
        device_key = get_device_key(record)
        if device_key not in cost_assumptions:
            continue

        costs = cost_assumptions[device_key]
        throughput = record.get('throughput_mean', 0)
        variant, config_value = device_key

        hw_config = f"CPU {config_value} cores" if variant == 'cpu' else f"GPU {config_value}%"

        processed_data.append({
            'hardware_config': hw_config,
            'variant': variant,
            'batch_size': record.get('concurrent_requests', 1),
            'throughput': throughput,
            'hourly_cost': costs['hourly_cost'],
            'idle_multiplier': costs['idle_multiplier'],
        })
    except Exception as e:
        continue

df = pd.DataFrame(processed_data)
print(f"Processed {len(df)} records for workload analysis")
print(f"Hardware configurations: {sorted(df['hardware_config'].unique())}")

In [ ]:
# Calculate workload metrics for 10 requests/hour scenario
def calculate_workload_metrics(df, requests_per_hour=10, avg_tokens_per_request=100):
    results = []

    for hw_config in df['hardware_config'].unique():
        config_data = df[df['hardware_config'] == hw_config]
        avg_throughput = config_data['throughput'].mean()
        hourly_cost = config_data['hourly_cost'].iloc[0]
        idle_multiplier = config_data['idle_multiplier'].iloc[0]
        variant = config_data['variant'].iloc[0]

        # Calculate processing time per request
        time_per_request = avg_tokens_per_request / avg_throughput  # seconds
        time_per_request_hours = time_per_request / 3600  # hours

        # Calculate total processing and idle time
        total_processing_time_hours = requests_per_hour * time_per_request_hours
        idle_time_hours = max(0, 1.0 - total_processing_time_hours)

        # Calculate costs
        processing_cost_per_hour = total_processing_time_hours * hourly_cost
        idle_cost_per_hour = idle_time_hours * hourly_cost * idle_multiplier
        total_hourly_cost = processing_cost_per_hour + idle_cost_per_hour
        cost_per_request = total_hourly_cost / requests_per_hour
        utilization_percent = (total_processing_time_hours / 1.0) * 100

        results.append({
            'hardware_config': hw_config,
            'variant': variant,
            'avg_throughput': avg_throughput,
            'cost_per_request': cost_per_request,
            'total_hourly_cost': total_hourly_cost,
            'utilization_percent': utilization_percent,
            'idle_cost_per_hour': idle_cost_per_hour,
            'processing_cost_per_hour': processing_cost_per_hour,
        })

    return pd.DataFrame(results)

# Calculate for 10 requests/hour
workload_10_rph = calculate_workload_metrics(df, requests_per_hour=10)
workload_sorted = workload_10_rph.sort_values('cost_per_request')

print("\n" + "="*70)
print("COST ANALYSIS: 10 REQUESTS/HOUR (100 tokens each)")
print("="*70)
print(f"{'Hardware':<15} {'Cost/Req':<12} {'Total/Hr':<10} {'Utilization':<12}")
print("-" * 70)

for _, row in workload_sorted.iterrows():
    print(f"{row['hardware_config']:<15} "
          f"${row['cost_per_request']:<11.4f} "
          f"${row['total_hourly_cost']:<9.3f} "
          f"{row['utilization_percent']:<11.1f}%")

# Show CPU advantage
best_cpu = workload_sorted[workload_sorted['variant'] == 'cpu'].iloc[0]
best_gpu = workload_sorted[workload_sorted['variant'] == 'cuda'].iloc[0]
cpu_advantage = (best_gpu['cost_per_request'] / best_cpu['cost_per_request'] - 1) * 100

print(f"\n🏆 CPU ADVANTAGE:")
print(f"Best CPU: {best_cpu['hardware_config']} - ${best_cpu['cost_per_request']:.4f}/request")
print(f"Best GPU: {best_gpu['hardware_config']} - ${best_gpu['cost_per_request']:.4f}/request")
print(f"CPU is {cpu_advantage:.1f}% cheaper per request!")

In [ ]:
# Create visualizations
configs = sorted(df['hardware_config'].unique())
cpu_configs = [c for c in configs if c.startswith('CPU')]
gpu_configs = [c for c in configs if c.startswith('GPU')]

cpu_colors = plt.cm.Oranges(np.linspace(0.4, 0.9, len(cpu_configs)))
gpu_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(gpu_configs)))

color_map = {}
for i, config in enumerate(cpu_configs):
    color_map[config] = cpu_colors[i]
for i, config in enumerate(gpu_configs):
    color_map[config] = gpu_colors[i]

# Cost per Request Visualization
plt.figure(figsize=(14, 8))
configs_ordered = workload_sorted['hardware_config'].tolist()
costs = workload_sorted['cost_per_request'].tolist()
colors = [color_map[config] for config in configs_ordered]

bars = plt.bar(range(len(configs_ordered)), costs, color=colors, alpha=0.8, edgecolor='black')

# Add value labels
for i, (bar, cost) in enumerate(zip(bars, costs)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(costs)*0.01,
             f'${cost:.4f}', ha='center', va='bottom', fontweight='bold')

plt.xlabel('Hardware Configuration', fontsize=14)
plt.ylabel('Cost per Request ($)', fontsize=14)
plt.title('Cost per Request Analysis\n10 Requests/Hour, Q4_K_M Model', fontsize=16, fontweight='bold')
plt.xticks(range(len(configs_ordered)), configs_ordered, rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')

# Highlight CPU advantage zone
cpu_indices = [i for i, config in enumerate(configs_ordered) if config.startswith('CPU')]
if cpu_indices:
    plt.axhspan(0, costs[cpu_indices[-1]], alpha=0.1, color='orange', label='CPU Advantage Zone')
    plt.legend()

plt.tight_layout()
plt.show()

print("\n📊 Key Insight: CPUs dominate in low-utilization scenarios!")
print("At 10 requests/hour, hardware utilization is <1%, making CPU's lower costs advantageous.")

In [ ]:
# Break-even Analysis
request_volumes = [1, 5, 10, 20, 50, 100, 200, 500]
breakeven_data = []

for rph in request_volumes:
    workload_result = calculate_workload_metrics(df, requests_per_hour=rph)
    for _, row in workload_result.iterrows():
        breakeven_data.append({
            'requests_per_hour': rph,
            'hardware_config': row['hardware_config'],
            'variant': row['variant'],
            'cost_per_request': row['cost_per_request'],
            'utilization_percent': row['utilization_percent']
        })

breakeven_df = pd.DataFrame(breakeven_data)

# Break-even Visualization
plt.figure(figsize=(16, 8))
for hw_config in sorted(df['hardware_config'].unique()):
    config_data = breakeven_df[breakeven_df['hardware_config'] == hw_config]
    plt.plot(config_data['requests_per_hour'], config_data['cost_per_request'],
            marker='o', linewidth=3, markersize=8,
            label=hw_config, color=color_map[hw_config])

plt.xlabel('Requests per Hour', fontsize=14)
plt.ylabel('Cost per Request ($)', fontsize=14)
plt.title('Break-even Analysis: Cost per Request vs Request Volume', fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.yscale('log')

plt.axvline(x=10, color='red', linestyle='--', alpha=0.7, label='Current Analysis')
plt.tight_layout()
plt.show()

# Summary of break-even points
print("\n" + "="*50)
print("BREAK-EVEN ANALYSIS SUMMARY")
print("="*50)

for rph in [10, 50, 100, 500]:
    volume_data = breakeven_df[breakeven_df['requests_per_hour'] == rph].sort_values('cost_per_request')
    if len(volume_data) > 0:
        best = volume_data.iloc[0]
        print(f"At {rph:3} req/hr: {best['hardware_config']:15} ${best['cost_per_request']:.4f}/req ({best['utilization_percent']:.1f}% util)")

In [ ]:
# Final Summary
print("\n" + "="*80)
print("WORKLOAD COST ANALYSIS SUMMARY")
print("="*80)

print(f"\nScenario: 10 requests/hour, 100 tokens per request")
print(f"Model: Q4_K_M Quantization")
print(f"Total workload: 1,000 tokens/hour")

# Calculate monthly and annual costs
best_cpu_monthly = best_cpu['total_hourly_cost'] * 24 * 30
best_gpu_monthly = best_gpu['total_hourly_cost'] * 24 * 30
monthly_savings = best_gpu_monthly - best_cpu_monthly

print(f"\n💰 COST COMPARISON:")
print(f"Best CPU ({best_cpu['hardware_config']}):")
print(f"  Cost per request: ${best_cpu['cost_per_request']:.4f}")
print(f"  Monthly cost: ${best_cpu_monthly:.0f}")
print(f"  Utilization: {best_cpu['utilization_percent']:.2f}%")

print(f"\nBest GPU ({best_gpu['hardware_config']}):")
print(f"  Cost per request: ${best_gpu['cost_per_request']:.4f}")
print(f"  Monthly cost: ${best_gpu_monthly:.0f}")
print(f"  Utilization: {best_gpu['utilization_percent']:.2f}%")

print(f"\n🎯 KEY INSIGHTS:")
print(f"• CPU saves ${monthly_savings:.0f}/month (${monthly_savings*12:.0f}/year) at low utilization")
print(f"• At <1% utilization, idle costs dominate - CPUs win")
print(f"• GPUs become cost-effective only at higher request volumes")
print(f"• Break-even point likely around 100-200 requests/hour")

# Save results
workload_10_rph.to_csv('workload_cost_analysis_results.csv', index=False)
print(f"\nResults saved to 'workload_cost_analysis_results.csv'")